In [ ]:
## Experiment Part 2

import random
import numpy as np
import torch
import monai

def set_seed(seed=42):
    """Locks all random number generators for exact reproducibility."""
    # 1. Set Python and Numpy seeds
    random.seed(seed)
    np.random.seed(seed)
    
    # 2. Set PyTorch seeds
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
        # Force cuDNN to behave deterministically (might slightly slow down training)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # 3. Set MONAI's internal seed for its randomized transforms
    monai.utils.set_determinism(seed=seed)

# Call the function immediately to lock everything in
set_seed(42)
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, ScaleIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([ScaleIntensity(), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(5, 7), magnitude_range=(10, 50),spatial_size=(96, 96, 96)),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([ScaleIntensity(), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=8, num_workers=8, pin_memory=torch.cuda.is_available())

# Create DenseNet121, CrossEntropyLoss and Adam optimizer
model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2).to(device)

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 150

patience = 20            # Stop after 20 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    val_loss_sum = 0.0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            val_loss = loss_function(val_outputs, val_labels)
            val_loss_sum += val_loss.item()
            value = torch.eq(val_outputs.argmax(dim=1), val_labels)
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    avg_val_loss = val_loss_sum / len(val_loader)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_densenet121_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, ScaleIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([ScaleIntensity(), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(6, 8), magnitude_range=(10, 50),spatial_size=(96, 96, 96)),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([ScaleIntensity(), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=8, num_workers=8, pin_memory=torch.cuda.is_available())

# Create DenseNet121, CrossEntropyLoss and Adam optimizer
model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2).to(device)

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 150

patience = 20            # Stop after 20 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    val_loss_sum = 0.0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            val_loss = loss_function(val_outputs, val_labels)
            val_loss_sum += val_loss.item()
            value = torch.eq(val_outputs.argmax(dim=1), val_labels)
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    avg_val_loss = val_loss_sum / len(val_loader)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_densenet121_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, ScaleIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([ScaleIntensity(), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(6, 8), magnitude_range=(10, 50),spatial_size=(96, 96, 96)),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([ScaleIntensity(), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=8, num_workers=8, pin_memory=torch.cuda.is_available())

# Create DenseNet121, CrossEntropyLoss and Adam optimizer
model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2).to(device)

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # We want the validation loss to go DOWN
    factor=0.5,        # Multiply LR by 0.5 when plateauing
    patience=5,        # Wait 5 epochs before dropping LR
    min_lr=1e-7        # Absolute minimum LR
)

# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 150

patience = 20            # Stop after 20 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            value = torch.eq(val_outputs.argmax(dim=1), val_labels)
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_densenet121_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()